## 📝 실전 문제풀이 Set 4: 디지털프라자 매출 데이터

### 1) 데이터 및 시나리오

▶ 디지털프라자 A지점에서 연휴 직전에 대대적인 할인 행사를 진행했다. 예상보다 많은 손님이 방문했고, 발생한 매출 데이터를 취합하여 향후 발송할 판촉물 컨텐츠를 기획하고자 한다. (1행 = 물품 1개 구매 내역)

* **데이터 개요:** `Xa sales_pos.csv` (550068 rows, 11 columns, UTF-8)

**[변수 상세]**

| 변수명 | 유형 | 설명 |
| --- | --- | --- |
| `user` | int | 고객 식별자 |
| `prod` | string | 상품 식별자 |
| `gender` | string | 성별 |
| `age_group` | string | 연령대 |
| `job` | int | 직업 구분 |
| `city` | string | 도시 유형 구분 |
| `marital` | int | 결혼 여부(1: 결혼) |
| `prod_cat1` | int | 상품 카테고리(1차) |
| `prod_cat2` | int | 상품 카테고리(2차) |
| `prod_cat3` | int | 상품 카테고리(3차) |
| `purchase` | int | 결제 금액 |

## 2) 문제

* **필요 라이브러리:** `MinMaxScaler`, `KMeans`, `silhouette_score`


In [5]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv('../dataset/sales_pos.csv')
display(df)

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
0,1,P00069042,F,0-17,10,A,0,3,NaN,NaN,8370
1,1,P00248942,F,0-17,10,A,0,1,6.0,14.0,15200
2,1,P00087842,F,0-17,10,A,0,12,NaN,NaN,1422
3,1,P00085442,F,0-17,10,A,0,12,14.0,NaN,1057
4,2,P00285442,M,55+,16,C,0,8,NaN,NaN,7969
...,...,...,...,...,...,...,...,...,...,...,...
550063,6033,P00372445,M,51-55,13,B,1,20,NaN,NaN,368
550064,6035,P00375436,F,26-35,1,C,0,20,NaN,NaN,371
550065,6036,P00375436,F,26-35,15,B,1,20,NaN,NaN,137
550066,6038,P00375436,F,55+,1,C,0,20,NaN,NaN,365


### **Q01.** 상품별 매출액(purchase)을 합산하여 그 매출액이 가장 큰 상품을 확인하고, **해당 상품을 가장 많이 구매하는 직업(job)을 확인하시오.**

* ※ 직업 확인 시 상품 구매 개수를 기준으로 확인하시오. 결과는 job 변수의 번호를 출력. (정답 예시: 1)

In [27]:
df_q1 = df.copy()
df_q1_gb_idxmax = df_q1.groupby(['prod'])['purchase'].sum().idxmax()

df_q1_job = df_q1[df_q1['prod'] == df_q1_gb_idxmax]['job'].value_counts()
df_q1_job.idxmax()

4

In [29]:
df_q1_ = df.copy()
df_q1_gb_ = df_q1_.groupby(['prod'])
display(df_q1_gb_)
display(df_q1_gb_.groups)

df_q1_job_ = df_q1[df_q1['prod'] == df_q1_gb_idxmax]['job'].value_counts()
display(df_q1_job_, type(df_q1_job_))

{'P00000142': [293, 1066, 1073, 1322, 1910, 2805, 3014, 3551, 3742, 5226, 5316, 5882, 6854, 7383, 8489, 9913, 10875, 11224, 11521, 11724, 11922, 12105, 12937, 13370, 13725, 14294, 14660, 14848, 15191, 15341, 15822, 16443, 16758, 18837, 19312, 20001, 20617, 22103, 22486, 23474, 23773, 24296, 24398, 24403, 25936, 27288, 27503, 28515, 28667, 29271, 29281, 29897, 29915, 30259, 30419, 30522, 31085, 31354, 31364, 31494, 31604, 31909, 32043, 32215, 32477, 32796, 34067, 34326, 34469, 34493, 35024, 35033, 35650, 36015, 36412, 36790, 36845, 37358, 38116, 38700, 41479, 41554, 42105, 42486, 43545, 43569, 44278, 44596, 44650, 45684, 45841, 45940, 47423, 47518, 47642, 47861, 48148, 48582, 49109, 49185, ...], 'P00000242': [11232, 12424, 16544, 19188, 19599, 20254, 20648, 20775, 22832, 23731, 28496, 28597, 30545, 31603, 32581, 34398, 34730, 35095, 36713, 36758, 37631, 38805, 39415, 39499, 39905, 40479, 41396, 43532, 45970, 47946, 48195, 48540, 50794, 52168, 53156, 58239, 59106, 60763, 61620, 62514, 63

4     221
7     187
0     179
17    143
1     124
12    117
14     84
2      78
20     75
16     60
10     55
6      54
15     50
3      46
11     38
5      31
13     24
19     20
18     14
9      12
8       3
Name: job, dtype: int64

pandas.core.series.Series

### **Q02.** 결혼 여부(marital)에 따라 구매하는 물품 종류 차이를 확인하고자 한다. 비교적 신혼부부가 많은 26-35세 그룹을 대상으로 각 고객의 구매물품 카테고리 개수를 산출하고, **결혼여부별로 그 평균값의 차이를 산출하시오.**

* ※ 구매 물품 카테고리 개수 산출에는 prod_cat1, 2, 3을 사용하며 결측치는 0으로 대치한다.
* ※ 카테고리 처리 예시: `prod_cat1=1, prod_cat2=2, prod_cat3=0` -> `1-2-0`
* ※ 정답은 절대값을 반올림하여 소수점 둘째 자리까지 출력하시오. (정답 예시: 0.12)

In [69]:
df_q2 = df[df['age_group'] == '26-35'].copy()
display(df_q2)
display(df_q2['prod_cat1'].apply(type).value_counts())
display(df_q2['prod_cat2'].apply(type).value_counts())
display(df_q2['prod_cat3'].apply(type).value_counts())
display(df_q2['age_group'].apply(type).value_counts())
display(df_q2['age_group'].unique())

col = ['prod_cat1','prod_cat2','prod_cat3']
df_q2[col] = df_q2[col].fillna(0).astype(int).astype(str)

df_q2['comb'] = df_q2['prod_cat1'] + "-" + df_q2['prod_cat2'] + "-" + df_q2['prod_cat3']
display(df_q2['comb'].unique())
display(df_q2['comb'].nunique())
display(df_q2['comb'].apply(type).value_counts())
display(df_q2)

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
5,3,P00193542,M,26-35,15,A,0,1,2.0,NaN,15227
9,5,P00274942,M,26-35,20,A,1,8,NaN,NaN,7871
10,5,P00251242,M,26-35,20,A,1,5,11.0,NaN,5254
11,5,P00014542,M,26-35,20,A,1,8,NaN,NaN,3957
12,5,P00031342,M,26-35,20,A,1,8,NaN,NaN,6073
...,...,...,...,...,...,...,...,...,...,...,...
550058,6024,P00372445,M,26-35,12,A,1,20,NaN,NaN,121
550059,6025,P00370853,F,26-35,1,B,0,19,NaN,NaN,48
550061,6029,P00372445,F,26-35,1,C,1,20,NaN,NaN,599
550064,6035,P00375436,F,26-35,1,C,0,20,NaN,NaN,371


<class 'int'>    219587
Name: prod_cat1, dtype: int64

<class 'float'>    219587
Name: prod_cat2, dtype: int64

<class 'float'>    219587
Name: prod_cat3, dtype: int64

<class 'str'>    219587
Name: age_group, dtype: int64

array(['26-35'], dtype=object)

array(['1-2-0', '8-0-0', '5-11-0', '1-2-5', '1-5-15', '5-14-0', '1-8-14',
       '6-8-0', '5-8-14', '8-17-0', '8-13-0', '1-6-0', '5-8-0', '1-8-0',
       '8-16-0', '1-16-0', '1-2-6', '1-6-14', '5-0-0', '5-15-0', '2-4-0',
       '5-8-18', '1-8-9', '2-0-0', '2-17-0', '13-16-0', '8-14-17',
       '4-8-9', '1-2-15', '1-15-16', '5-9-14', '7-0-0', '8-13-15',
       '16-0-0', '1-8-16', '1-6-15', '8-15-0', '4-5-9', '4-5-0', '15-0-0',
       '1-8-17', '1-15-0', '3-5-0', '1-6-8', '1-14-0', '1-2-16',
       '11-13-16', '18-0-0', '5-17-0', '1-14-17', '8-9-14', '1-11-15',
       '1-13-16', '1-6-16', '1-5-14', '3-15-0', '6-8-15', '11-15-0',
       '12-14-0', '3-4-5', '6-8-10', '6-8-13', '6-8-16', '1-2-8',
       '10-16-0', '3-4-12', '8-15-16', '15-16-0', '8-10-0', '1-2-13',
       '5-9-0', '1-2-3', '1-14-16', '1-0-0', '8-14-0', '5-6-9', '2-9-15',
       '1-2-9', '11-0-0', '5-12-0', '6-16-0', '1-2-14', '1-5-18',
       '10-15-16', '13-0-0', '1-2-11', '6-8-14', '2-8-14', '3-0-0',
       '8-14-16', '5-

235

<class 'str'>    219587
Name: comb, dtype: int64

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase,comb
5,3,P00193542,M,26-35,15,A,0,1,2,0,15227,1-2-0
9,5,P00274942,M,26-35,20,A,1,8,0,0,7871,8-0-0
10,5,P00251242,M,26-35,20,A,1,5,11,0,5254,5-11-0
11,5,P00014542,M,26-35,20,A,1,8,0,0,3957,8-0-0
12,5,P00031342,M,26-35,20,A,1,8,0,0,6073,8-0-0
...,...,...,...,...,...,...,...,...,...,...,...,...
550058,6024,P00372445,M,26-35,12,A,1,20,0,0,121,20-0-0
550059,6025,P00370853,F,26-35,1,B,0,19,0,0,48,19-0-0
550061,6029,P00372445,F,26-35,1,C,1,20,0,0,599,20-0-0
550064,6035,P00375436,F,26-35,1,C,0,20,0,0,371,20-0-0


In [105]:
df_q2_user_category_count = df_q2.groupby(['user', 'marital'])['comb'].nunique().reset_index(name='comb_count')
df_q2_marital = df_q2_user_category_count.groupby('marital')['comb_count'].mean()
display(df_q2_marital)
round(abs(df_q2_marital[0] - df_q2_marital[1]), 2)


marital
0    41.663183
1    41.792336
Name: comb_count, dtype: float64

0.13

### **Q03.** 고객 5891명을 군집화하여 마케팅 전략을 수립하고자 한다. 제시된 변수(성별, 구매 상품 종류수, 나이, 직업, 총 구매금액, 도시, 결혼 여부)를 대상으로 **k-means 군집분석(K=7)을 실시했을 때 Silhouette score를 산출하시오.**

* ※ 성별 변수는 M=1, F=0으로 변환, 나이는 순서형(0~6)으로 변환, 직업과 도시는 One Hot Encoding.
* ※ 사용 변수는 총 29개이며 MinMax 정규화 후 분석. seed는 123. (정답 예시: 0.12)

고객 5891명을 군집화하여 마케팅 전략을 수립하고자 한다. 제시된 변수(성별, 구매 상품 종류수, 나이, 직업, 총 구매금액, 도시, 결혼 여부)를 대상으로 **k-means 군집분석(K=7)을 실시했을 때 Silhouette score를 산출하시오.

-->

고객 5891명 별 제시된 변수(성별, 구매 상품 종류수, 나이, 직업, 총 구매금액, 도시, 결혼 여부)를 대상으로 ...
이므로 groupby가 떠오르게 되고, 저 변수들을 groupby 했을 때 그대로 쓰려면 agg로 활용한다.

**[변수 상세]**

| 변수명 | 유형 | 설명 |
| --- | --- | --- |
| `user` | int | 고객 식별자 |
| `prod` | string | 상품 식별자 |
| `gender` | string | 성별 |
| `age_group` | string | 연령대 |
| `job` | int | 직업 구분 |
| `city` | string | 도시 유형 구분 |
| `marital` | int | 결혼 여부(1: 결혼) |
| `prod_cat1` | int | 상품 카테고리(1차) |
| `prod_cat2` | int | 상품 카테고리(2차) |
| `prod_cat3` | int | 상품 카테고리(3차) |
| `purchase` | int | 결제 금액 |

In [204]:
df_q3 = df.groupby('user').agg(
    gender=('gender', 'first'),
    prod=('prod', 'nunique'),
    age_group=('age_group', 'first'),
    job=('job', 'first'),
    purchase=('purchase', 'sum'),
    city=('city', 'first'),
    marital=('marital', 'first') 
).reset_index() #groupby로 'user'가 index가 되니 'user'도 columns 으로 쓰기위함

display(df_q3.columns)
#gendor 변경
df_q3['gender'] = df_q3['gender'].replace({'M' : 1, 'F' : 0})

#age_group 변경
age_ = df_q3['age_group'].drop_duplicates().sort_values(ascending = True).reset_index(drop=True) #series 의 sorting
#display(age_)
age_dict = dict(zip(age_, age_.index))
#display(age_dict)
df_q3['age_group'] = df_q3['age_group'].replace(age_dict)


#modeling --
#D
#군집변수에는 user가 제외되어야 한다!!!
X_trained = pd.get_dummies(df_q3.drop(columns='user'), columns=['job','city'], drop_first=False)
display(X_trained.columns, len(X_trained.columns))

#N
scaler = MinMaxScaler()
X_trained_scaled = scaler.fit_transform(X_trained)
#M
model = KMeans(n_clusters = 7, random_state = 123, n_init = 10)
pred = model.fit_predict(X_trained_scaled)

#E
sil = silhouette_score(X_trained_scaled, pred)
display(sil)
round(sil, 2)

Index(['user', 'gender', 'prod', 'age_group', 'job', 'purchase', 'city',
       'marital'],
      dtype='object')

Index(['gender', 'prod', 'age_group', 'purchase', 'marital', 'job_0', 'job_1',
       'job_2', 'job_3', 'job_4', 'job_5', 'job_6', 'job_7', 'job_8', 'job_9',
       'job_10', 'job_11', 'job_12', 'job_13', 'job_14', 'job_15', 'job_16',
       'job_17', 'job_18', 'job_19', 'job_20', 'city_A', 'city_B', 'city_C'],
      dtype='object')

29

0.17879229751009443

0.18

In [158]:
display(df.columns)
df_q3 = df[['gender', 'prod', 'age_group', 'job','purchase','city','marital']].copy()
df_q3

display(df_q3['gender'].apply(type).value_counts(), df_q3['gender'].nunique(), df_q3['gender'].unique())
display(df_q3['age_group'].apply(type).value_counts(), df_q3['age_group'].nunique(), df_q3['age_group'].unique())
display(df_q3['job'].apply(type).value_counts(), df_q3['job'].nunique(), df_q3['job'].unique())
display(df_q3['city'].apply(type).value_counts(), df_q3['city'].nunique(), df_q3['city'].unique())

Index(['user', 'prod', 'gender', 'age_group', 'job', 'city', 'marital',
       'prod_cat1', 'prod_cat2', 'prod_cat3', 'purchase'],
      dtype='object')

<class 'str'>    550068
Name: gender, dtype: int64

2

array(['F', 'M'], dtype=object)

<class 'str'>    550068
Name: age_group, dtype: int64

...

21

array([10, 16, 15,  7, 20,  9,  1, 12, 17,  0,  3,  4, 11,  8, 19,  2, 18,
        5, 14, 13,  6], dtype=int64)

<class 'str'>    550068
Name: city, dtype: int64

3

array(['A', 'C', 'B'], dtype=object)

In [160]:
# 고객 군집화
df['user'].nunique()

5891

In [155]:
# 성별 바꾸기 --> map
df_q3['gender'] = df_q3['gender'].map({'M' : 1,'F' : 0})
display(df_q3)

#age 바꾸기
age_group = df_q3['age_group'].drop_duplicates().sort_values()
age_group_pd = pd.Series(range(len(age_group)), index = age_group)
display(age_group_pd)
df_q3['age_group'] = df_q3['age_group'].replace(age_group_pd)
display(df_q3)

#one - hot encoding
df_q3 = pd.get_dummies(df_q3, columns=['job', 'city'], drop_first = False)
display(df_q3)

df_q3.shape


,gender,prod,age_group,job,purchase,city,marital
0,0,P00069042,0-17,10,8370,A,0
1,0,P00248942,0-17,10,15200,A,0
2,0,P00087842,0-17,10,1422,A,0
3,0,P00085442,0-17,10,1057,A,0
4,1,P00285442,55+,16,7969,C,0
...,...,...,...,...,...,...,...
550063,1,P00372445,51-55,13,368,B,1
550064,0,P00375436,26-35,1,371,C,0
550065,0,P00375436,26-35,15,137,B,1
550066,0,P00375436,55+,1,365,C,0


age_group
0-17     0
18-25    1
26-35    2
36-45    3
46-50    4
51-55    5
55+      6
dtype: int64

,gender,prod,age_group,job,purchase,city,marital
0,0,P00069042,0,10,8370,A,0
1,0,P00248942,0,10,15200,A,0
2,0,P00087842,0,10,1422,A,0
3,0,P00085442,0,10,1057,A,0
4,1,P00285442,6,16,7969,C,0
...,...,...,...,...,...,...,...
550063,1,P00372445,5,13,368,B,1
550064,0,P00375436,2,1,371,C,0
550065,0,P00375436,2,15,137,B,1
550066,0,P00375436,6,1,365,C,0


,gender,prod,age_group,purchase,marital,job_0,job_1,job_2,job_3,job_4,...,job_14,job_15,job_16,job_17,job_18,job_19,job_20,city_A,city_B,city_C
0,0,P00069042,0,8370,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0,P00248942,0,15200,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,P00087842,0,1422,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,P00085442,0,1057,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,1,P00285442,6,7969,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550063,1,P00372445,5,368,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
550064,0,P00375436,2,371,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
550065,0,P00375436,2,137,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
550066,0,P00375436,6,365,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1


(550068, 29)

In [157]:
#DNME

#D
X = df_q3

#N
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

#M
model = KMeans(n_clusters = 7, random_state = 123)
cluster = model.fit(X_scaled)

#E
sil = silhouette_score(X_scaled, cluster)
round(sil, 2)


ValueError: could not convert string to float: 'P00069042'